In [1]:
# file_path = r"/Users/suppi/santosh/medical_extraction/files/SANTOSH KUMAR K.pdf"
file_path = r"/Users/ksk/Desktop/medical_extraction/files/10.pdf"

In [2]:
! pip install pdf2image
! pip install -q -U google-genai Pillow langchain_google_genai PyPDF2

In [3]:
import pandas as pd
import re
def markdown_to_dataframe(markdown_text):
    '''
    Input : pdf path => str
    description : creating 2 page chunks
    Output : list of chunks => list
    '''
    table_pattern = r"((?:\|.+\|(?:\n|\r))+\|.*\|)"
    tables = re.findall(table_pattern, markdown_text)
    combined_df = pd.DataFrame()
    for table in tables:
        rows = table.strip().split("\n")
        headers = rows[0].strip("|").split("|")
        headers = [h.strip() for h in headers]
        data = []
        len_records = []
        for row in rows[2:]:
            values = row.strip("|").split("|")
            values = [v.strip() for v in values]
            data.append(values)
            len_records.append(len(values))
        df = pd.DataFrame(data, columns=headers)
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    return combined_df

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.0-flash",
#     temperature=0,
#     max_tokens=10
# )

from langchain_google_genai import GoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, max_tokens=100)

In [12]:
llm.invoke("Hi hello").content

'Hi there'

In [5]:
import os
from PyPDF2 import PdfReader
df = pd.DataFrame()
folder_path = r"./files"
items = os.listdir(folder_path)
for item in items:
    file_path = os.path.join(folder_path, item)
    reader = PdfReader(file_path)
    pages = reader.pages
    document = []
    for i in range(len(pages)):
        document.append(pages[i].extract_text())
    document = "\n".join(document)
    prompt = f"""
You are expert in extracting data from medical records. Your task is to extract the bio marker values from the medical records in a table format.
Medical Records : {document}
*The output should be only markdown table.*
*Each column name should be a bio-marker name, the column value should be bio-marker value.*
*The first column should be report date and next column should be doctor name*
*The date column should always be in mm-dd-yyyy format*
"""
    ans = llm.invoke(prompt).content
    df_gemini = markdown_to_dataframe(ans)
    df = pd.concat([df, df_gemini])

PdfReadError: EOF marker not found

In [ ]:
df

In [ ]:
column_names = ["SGPT", "Alanine Transaminase", "ALT", "HbA1c", "C reactive protein"]
prompt_normalization = f"""
You are a medical data expert. Normalize the following column names to standard biomarker terms (like ALT, HbA1c, CRP, creatinine etc.):
{column_names}
Return a JSON dictionary of the form: "original_name": "standard_name"
"""
ans = llm.invoke(prompt).content


In [ ]:
print(ans)

In [ ]:
ans = ans.strip("```").strip("json")

In [ ]:
ans

In [ ]:
import json
s = json.loads(ans)

In [ ]:
type(s)

In [ ]:
def normalize_column_names_llm(columns, llm):
    prompt = f"""
You are a medical data expert. Normalize the following column names to standard biomarker names such as ALT, AST, HbA1c, CRP, creatinine, bilruben ,etc. Use common medical conventions.

Column names: {columns}
Make sure the names are generic.
Return a JSON dictionary mapping each original name to its normalized name.
"""
    response = llm.invoke(prompt).content
    # Clean up the response
    cleaned = response.replace("```json", "").replace("```", "").strip()
    print("LLM normalization response:", cleaned) 
    try:
        mapping = json.loads(cleaned)
        return mapping
    except Exception as e:
        print("Failed to parse LLM normalization response:", e)
        return {}

In [ ]:
# Global cache for column mappings
COLUMN_MAPPING_CACHE = {}

def normalize_column_names_llm(columns, llm):
    # Filter out already cached columns
    uncached_columns = [col for col in columns if col not in COLUMN_MAPPING_CACHE]
    
    if not uncached_columns:
        return COLUMN_MAPPING_CACHE
    
    # MODIFIED PROMPT - more explicit instructions
    prompt = f"""
You are a medical data standardization expert. Convert the following lab report column names to standard biomarker abbreviations using international nomenclature guidelines.

Rules:
1. Use only standard abbreviations (e.g., ALT, AST, HbA1c)
2. Remove units, reference ranges, and method information
3. Convert similar names to single standard (e.g., "HbA1c", "A1C" → "HbA1c")
4. For non-biomarker columns (like patient IDs), return "ignore"
5. Maintain case sensitivity (all caps for biomarkers)

Column names to normalize: {uncached_columns}

Return ONLY a JSON dictionary where:
- Key = original column name
- Value = normalized standard name or "ignore"
"""
    try:
        response = llm.invoke(prompt).content
        cleaned = response.replace("```json", "").replace("```", "").strip()
        new_mappings = json.loads(cleaned)
        
        # Cache new mappings and filter invalid
        for orig, normalized in new_mappings.items():
            if normalized.lower() != "ignore":
                COLUMN_MAPPING_CACHE[orig] = normalized
                
        return COLUMN_MAPPING_CACHE
    
    except Exception as e:
        print(f"Normalization failed: {e}")
        # Fallback to original names
        return {col: col for col in columns}


In [ ]:
import os
from PyPDF2 import PdfReader
import pandas as pd
import json

df = pd.DataFrame()
folder_path = r"./files"
# folder_path = r"./test"
items = os.listdir(folder_path)

def normalize_column_names_llm(columns, llm):
    prompt = f"""
You are a medical data expert. Normalize the following column names to standard biomarker names such as ALT, AST, HbA1c, CRP, creatinine, bilruben ,etc. Use common medical conventions.

Column names: {columns}
Make sure the names are generic.
Ignore patient names, report dates, and doctor names.
Return a JSON dictionary mapping each original name to its normalized name.
"""
    response = llm.invoke(prompt).content
    # Clean up the response
    cleaned = response.replace("```json", "").replace("```", "").strip()
    print("LLM normalization response:", cleaned)  # Debug print
    try:
        mapping = json.loads(cleaned)
        return mapping
    except Exception as e:
        print("Failed to parse LLM normalization response:", e)
        return {}


for item in items:
    file_path = os.path.join(folder_path, item)
    reader = PdfReader(file_path)
    pages = reader.pages
    document = [pages[i].extract_text() for i in range(len(pages))]
    document = "\n".join(document)
    # Generate table using LLM
    prompt = f"""
You are expert in extracting data from medical records. Your task is to extract the bio marker values from the medical records in a table format.

Medical Records : {document}

*The output should be only markdown table.*
*Each column name should be a bio-marker name, the column value should be bio-marker value.*
*The first column should be *patient_name* followed by  *report_date* and then *doctor_name*.*
*The date column should always be in mm-dd-yyyy format.*
*Donot include units in the values*
"""
    
    ans = llm.invoke(prompt).content
    df_gemini = markdown_to_dataframe(ans)

    # Normalize biomarker columns using LLM
    columns_to_normalize = [col for col in df_gemini.columns if col not in ["report_date", "doctor_name"]]
    column_mapping = normalize_column_names_llm(columns_to_normalize, llm)
    
    # Apply mapping
    df_gemini = df_gemini.rename(columns=column_mapping)

    # Append to final df
    df = pd.concat([df, df_gemini], ignore_index=True)


# Working 

In [3]:
prompt_extraction_table = """
You are expert in extracting data from medical records. Your task is to extract the bio marker values from the medical records in a table format.

Medical Records : {document}

*The output should be only markdown table.*
*Each column name should be a bio-marker name, the column value should be bio-marker value.*
*The first column should be *patient_name* followed by  *report_date*, *lab_name(Lab Name or Diagnostic centre name)* *and then *doctor_name*.*
*The date column should always be in mm-dd-yyyy format.*
If there is nothing to extract Fill that field with *N/A*.
*Donot include units in the values*
"""


In [4]:
normalization_prompt = """
You are a medical data expert. Normalize the following column names to standard biomarker names such as ALT, AST, HbA1c, CRP, creatinine, bilruben ,etc. Use common medical conventions.
Column names: {columns}
Make sure the names are generic.
Return a JSON dictionary mapping each original name to its normalized name.
"""


In [5]:
import pandas as pd
import re
def markdown_to_dataframe(markdown_text):
    '''
    Input : pdf path => str
    description : creating 2 page chunks
    Output : list of chunks => list
    '''
    table_pattern = r"((?:\|.+\|(?:\n|\r))+\|.*\|)"
    tables = re.findall(table_pattern, markdown_text)
    combined_df = pd.DataFrame()
    for table in tables:
        rows = table.strip().split("\n")
        headers = rows[0].strip("|").split("|")
        headers = [h.strip() for h in headers]
        data = []
        len_records = []
        for row in rows[2:]:
            values = row.strip("|").split("|")
            values = [v.strip() for v in values]
            data.append(values)
            len_records.append(len(values))
        df = pd.DataFrame(data, columns=headers)
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    return combined_df

In [6]:
global_column_mapping = {}

def normalize_column_names_llm(columns, llm):
    """Normalizes columns with LLM, returns mapping with fallback to original names"""
    prompt = normalization_prompt.format(columns=columns)
    
    try:
        response = llm.invoke(prompt).content
        cleaned = response.replace("```json", "").replace("```", "").strip()
        new_mapping = json.loads(cleaned)
        
        # Validate and filter the response
        validated_mapping = {
            orig: normalized if normalized.lower() != "ignore" else orig
            for orig, normalized in new_mapping.items()
            if orig in columns  # Only keep mappings for requested columns
        }
        return validated_mapping
        
    except Exception as e:
        print(f"Normalization failed for columns {columns}: {e}")
        return {col: col for col in columns}  # Fallback to original names


In [7]:
from pdf2image import convert_from_path
from PIL import Image
import io
import base64
from langchain_google_genai import ChatGoogleGenerativeAI

def extract_labname_from_pdf(pdf_path, page_num=0):
    # Convert the specified page to image
    images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1)
    img = images[0]
    
    # Optional: Upscale for better clarity
    scale_factor = 3
    width, height = img.size
    img = img.resize((width * scale_factor, height * scale_factor), Image.LANCZOS)
    
    # Encode image to base64
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    img_bytes = buffer.getvalue()
    base64_image = base64.b64encode(img_bytes).decode("utf-8")
    
    # LLM prompt for lab name extraction
    prompt = (
        "Extract ONLY the lab name or diagnostic centre name from the given image. "
        "Return just the lab name as a plain string, nothing else."
        "If there is nothing to extract return N/A"
    )
    llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
            ]
        }
    ]
    resp = llm.invoke(messages).content
    # Clean up response if needed
    lab_name = resp.strip().split("\n")[0]
    return lab_name

ModuleNotFoundError: No module named 'pdf2image'

In [46]:
# conda install -c conda-forge poppler

In [ ]:
import os
from PyPDF2 import PdfReader
import pandas as pd
import json
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
)
df = pd.DataFrame()
folder_path = r"./files"
# folder_path = r"./test"
items = os.listdir(folder_path)
global_column_mapping = {}

for item in items:
    file_path = os.path.join(folder_path, item)
    reader = PdfReader(file_path)
    document = "\n".join(page.extract_text() for page in reader.pages)
    prompt = prompt_extraction_table.format(document=document)
    ans = llm.invoke(prompt).content
    df_gemini = markdown_to_dataframe(ans)

    # --- Fill lab_name if empty ---
    if "lab_name" in df_gemini.columns:
        # Find rows where lab_name is empty or null
        mask = df_gemini["lab_name"].isnull() | (df_gemini["lab_name"].astype(str).str.strip() == "N/A")
        if mask.any():
            # Extract lab name from the first page image
            lab_name_val = extract_labname_from_pdf(file_path, page_num=0)
            df_gemini.loc[mask, "lab_name"] = lab_name_val

    columns_to_normalize = [
        col for col in df_gemini.columns 
        if col not in ["patient_name", "report_date","lab_name", "doctor_name"]
    ]

    new_mapping = normalize_column_names_llm(
        columns=[col for col in columns_to_normalize if col not in global_column_mapping],
        llm=llm
    )
    
    global_column_mapping.update(new_mapping)
    rename_map = {
        col: global_column_mapping.get(col, col)
        for col in df_gemini.columns
    }
    df_gemini = df_gemini.rename(columns=rename_map)
    df = pd.concat([df, df_gemini], ignore_index=True)


In [49]:
df

,patient_name,report_date,lab_name,doctor_name,WBC,RBC,Hemoglobin,Hematocrit,MCV,MCH,...,Calcium,Liver Size,Spleen Size,Prostate Size,Right Kidney Size,Right Kidney Cortical Thickness,Left Kidney Size,Left Kidney Cortical Thickness,Transplant Kidney Ureter Thickness,Transplant Kidney Ureter Length
0,MR. SANTOSH KUMAR K,08-17-2023,AMPATH,DR.SEERAPANI GOPALUNI,4.4,3.8,12.1,34.6,90.5,31.7,...,9.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Mr. Santosh Kumar K,05-03-2023,N/A,Dr. Seerapani Gopaluni,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,12.9,9.9,17,4.5 x 2.3,6.0,5.9 x 3.0,7.0,3.4,2.8


In [48]:
df.lab_name


0    AMPATH
1       N/A
Name: lab_name, dtype: object

In [40]:
df.drop_duplicates()

,patient_name,report_date,lab_name,doctor_name,WBC,RBC,Hemoglobin,Hematocrit,MCV,MCH,...,Calcium,Liver Size,Spleen Size,Prostate Size,Right Kidney Size,Right Kidney Cortical Thickness,Left Kidney Size,Left Kidney Cortical Thickness,Transplant Kidney Ureter Thickness,Transplant Kidney Ureter Length
0,MR. SANTOSH KUMAR K,08-17-2023,AMPATH,DR.SEERAPANI GOPALUNI,4.4,3.8,12.1,34.6,90.5,31.7,...,9.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Mr. Santosh Kumar K,05-03-2023,N/A,Dr. Seerapani Gopaluni,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,12.9,9.9,17,4.5 x 2.3,6.0,5.9 x 3.0,7.0,3.4,2.8


In [41]:
print(list(df.columns))

['patient_name', 'report_date', 'lab_name', 'doctor_name', 'WBC', 'RBC', 'Hemoglobin', 'Hematocrit', 'MCV', 'MCH', 'MCHC', 'RDW', 'Platelet Count', 'MPV', 'Neutrophils', 'Lymphocytes', 'Monocytes', 'Eosinophils', 'Basophils', 'Absolute Neutrophil Count', 'Absolute Lymphocyte Count', 'Absolute Monocyte Count', 'Absolute Eosinophil Count', 'Absolute Basophil Count', 'Bilirubin Total', 'Bilirubin Direct', 'Bilirubin Indirect', 'ALT', 'AST', 'ALP', 'GGT', 'Total Protein', 'Albumin', 'Globulin', 'Albumin/Globulin Ratio', 'LDH', 'BUN', 'Uric Acid', 'Creatinine', 'Sodium', 'Potassium', 'Chloride', 'Calcium', 'Liver Size', 'Spleen Size', 'Prostate Size', 'Right Kidney Size', 'Right Kidney Cortical Thickness', 'Left Kidney Size', 'Left Kidney Cortical Thickness', 'Transplant Kidney Ureter Thickness', 'Transplant Kidney Ureter Length']


# New Check

In [ ]:

def normalize_column_names_llm(columns, llm):
    """Handles empty columns and invalid JSON responses"""
    if not columns:
        return {}

    prompt = normalization_prompt.format(columns=columns)
    
    try:
        response = llm.invoke(prompt).content
        cleaned = response.replace("```json", "").replace("```", "").strip()
        
        if not cleaned:
            print("Warning: LLM returned empty response")
            return {}
            
        return json.loads(cleaned)
        
    except json.JSONDecodeError:
        print(f"LLM returned invalid JSON for columns: {columns}")
        return {}
    except Exception as e:
        print(f"Unexpected error: {e}")
        return {}

In [ ]:
import os
from PyPDF2 import PdfReader
import pandas as pd
import json
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
)
df = pd.DataFrame()
folder_path = r"./files"
# folder_path = r"./test"
items = os.listdir(folder_path)
global_column_mapping = {}
for item in items:
    file_path = os.path.join(folder_path, item)
    reader = PdfReader(file_path)
    document = "\n".join(page.extract_text() for page in reader.pages)
    prompt = prompt_extraction_table.format(document=document)
    ans = llm.invoke(prompt).content
    df_gemini = markdown_to_dataframe(ans)
    columns_to_normalize = [
        col for col in df_gemini.columns 
        if col not in ["patient_name", "report_date", "doctor_name"]
    ]
    
    if columns_to_normalize:  # Only call LLM if needed
        new_mapping = normalize_column_names_llm(
            columns=[col for col in columns_to_normalize 
                    if col not in global_column_mapping],
            llm=llm
        )
        global_column_mapping.update(new_mapping)
    
    # Safe rename (works even if mapping is empty)
    df_gemini = df_gemini.rename(columns=lambda x: global_column_mapping.get(x, x))
    df = pd.concat([df, df_gemini], ignore_index=True)
    df.drop_duplicates(inplace=True)
    
# Post Processing
df.drop_duplicates(inplace=True)
df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")
df = df.sort_values("report_date", ascending=True)

In [ ]:
df["t"]

In [ ]:
q = df[['report_date', "Tacrolimus"]]

In [ ]:
x = df[df["doctor_name"] == "Ganta Swarna Rathan"]

In [ ]:
x.values

In [ ]:
z = x.to_dict()

In [ ]:
for k,v in z.items():
    print(k,v)

## Prescription

In [64]:
def upscale_image(image_path, scale_factor=2):
    """Upscales the image to improve text clarity."""
    img = Image.open(image_path)
    width, height = img.size
    img_resized = img.resize((width * scale_factor, height * scale_factor), Image.LANCZOS)

    buffer = io.BytesIO()
    img_resized.save(buffer, format="PNG")
    return buffer.getvalue()

def encode_image_to_base64(image_bytes):
    """Encodes image bytes to base64 string."""
    return base64.b64encode(image_bytes).decode("utf-8")

def solve_captcha(image_path):
    """Encodes an image file to a base64 string."""
    llm_gemini = ChatGoogleGenerativeAI(
            model="gemini-2.0-flash",
            temperature=0,
    )
    upscaled_image_bytes = upscale_image(image_path, scale_factor=4)
    base64_image = encode_image_to_base64(upscaled_image_bytes)
    # prompt = "Extract all the text from the given Image. The text in the image contains only alphabets and numbers. It does not contain anything apart from numbers and alphabets. *Return just the string the image in the markdown format*" 
    prompt = """You are an expert medical assistant.Extract all meaningful and relevant information from the following doctor's prescription text. The data may include (but is not limited to): patient details, doctor information, date, clinic name, medicine names, dosages, frequency, instructions, and any other relevant notes.Return the extracted data in a structured and readable JSON format. Group similar information logically, such as medicines under a "medicines" array, patient info under "patient", etc. Do not make up data. If any field is unclear or missing, leave it out.Only return the JSON object as output.
"""
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            ]
        }
    ]
    resp = llm_gemini.invoke(messages).content
    print(resp)
    return resp

In [65]:
img_path = r"/Users/suppi/Desktop/santosh/santosh/medical_extraction/files/WhatsApp Image 2025-07-08 at 23.15.24.jpeg"
prescription = solve_captcha(img_path)
prescription

```json
{
  "clinic": {
    "name": "CITIZENS SPECIALTY HOSPITAL"
  },
  "doctor": {
    "name": "Dr. Gopaluni Seerapani",
    "qualifications": "MBBS, MRCP(UK), MRCP (Nephrology), CCT(UK), Ph.D.(Cambridge)",
    "speciality": "Consultant Nephrologist & Transplant Physician",
    "other": "Specialist in Autoimmune Disease & Immuno-Nephrology",
    "reg_no": "49196"
  },
  "patient": {
    "name": "Mr. SANTOSH KUMAR",
    "gender": "Male",
    "age": "25"
  },
  "date": "5/7/25",
  "vitals": {
    "height": "cms",
    "weight": "kgs",
    "bp": "mmHg",
    "temp": "of",
    "pulse": "b/mt",
    "resp": "b/mt",
    "spo2": ""
  },
  "assessment": {
    "pain_scoring": {
      "mild_to_moderate": "Score(0-6)",
      "severe_discomfort_pain": "Score(7-10)"
    },
    "nutritional_assessment": {
      "normal_nutrition_status": "Score(10-12)",
      "refer_to_dietician_history": "Score<10"
    },
    "physical_examination": [
      "Doing well",
      "no fevers",
      "Good appetite",
   

'```json\n{\n  "clinic": {\n    "name": "CITIZENS SPECIALTY HOSPITAL"\n  },\n  "doctor": {\n    "name": "Dr. Gopaluni Seerapani",\n    "qualifications": "MBBS, MRCP(UK), MRCP (Nephrology), CCT(UK), Ph.D.(Cambridge)",\n    "speciality": "Consultant Nephrologist & Transplant Physician",\n    "other": "Specialist in Autoimmune Disease & Immuno-Nephrology",\n    "reg_no": "49196"\n  },\n  "patient": {\n    "name": "Mr. SANTOSH KUMAR",\n    "gender": "Male",\n    "age": "25"\n  },\n  "date": "5/7/25",\n  "vitals": {\n    "height": "cms",\n    "weight": "kgs",\n    "bp": "mmHg",\n    "temp": "of",\n    "pulse": "b/mt",\n    "resp": "b/mt",\n    "spo2": ""\n  },\n  "assessment": {\n    "pain_scoring": {\n      "mild_to_moderate": "Score(0-6)",\n      "severe_discomfort_pain": "Score(7-10)"\n    },\n    "nutritional_assessment": {\n      "normal_nutrition_status": "Score(10-12)",\n      "refer_to_dietician_history": "Score<10"\n    },\n    "physical_examination": [\n      "Doing well",\n      

In [77]:
prescription = prescription.replace("json","")
prescription = prescription.strip("```")
prescription_json = json.loads(prescription)

In [78]:
prescription_json.keys()

dict_keys(['clinic', 'doctor', 'patient', 'date', 'vitals', 'assessment', 'diagnosis', 'plan', 'allergies', 'notes'])

In [80]:
prescription_json["assessment"]["physical_examination"]

['Doing well',
 'no fevers',
 'Good appetite',
 'wt stable',
 'no urinary symptoms',
 'Bowels @',
 'Enjoying work']

In [83]:
prescription_json["diagnosis"]

{'provisional_final_diagnosis': 'Stable graft function'}

In [85]:
prescription_json["plan"]

['Tac levels awaited', 'No change in B', 'DEXA scan']

In [86]:
img_path = r"/Users/suppi/Desktop/santosh/santosh/medical_extraction/files/WhatsApp Image 2025-07-08 at 23.54.25.jpeg"
resp = solve_captcha(img_path)
resp

```json
{
  "clinic": {
    "name": "N.M.REDDY SKIN HAIR LASER & COSMETIC SURGERY CLINIC",
    "address": "Shop No. 3, Door No. 5-85, Opposite Chenna Reddy Hospital, Manjeera Road, Chandanagar, Hyderabad - 50"
  },
  "doctors": [
    {
      "name": "Dr. N. Manohar Reddy",
      "degree": "M.D. (DVL)",
      "specialization": "Consultant Dermatologist, Venereologist & Dermato Surgeon",
      "registration_number": "64661"
    },
    {
      "name": "Dr. N. Swetha Reddy",
      "degree": "MBBS, FHRS",
      "specialization": "Skin, Cosmetology & Hair Transplant Surgeon",
      "registration_number": "75548"
    }
  ],
  "patient": {
    "name": "k. Leela Supreja",
    "age": "28",
    "sex": "F"
  },
  "date": "10/03/25",
  "medicines": [
    {
      "name": "Manika/clanit",
      "dosage": "6",
      "frequency": "3 times"
    },
    {
      "name": "Khuges"
    },
    {
      "name": "Achin N3",
      "form": "cap",
      "dosage": "604",
      "frequency": "1/2m"
    },
    {
      "

'```json\n{\n  "clinic": {\n    "name": "N.M.REDDY SKIN HAIR LASER & COSMETIC SURGERY CLINIC",\n    "address": "Shop No. 3, Door No. 5-85, Opposite Chenna Reddy Hospital, Manjeera Road, Chandanagar, Hyderabad - 50"\n  },\n  "doctors": [\n    {\n      "name": "Dr. N. Manohar Reddy",\n      "degree": "M.D. (DVL)",\n      "specialization": "Consultant Dermatologist, Venereologist & Dermato Surgeon",\n      "registration_number": "64661"\n    },\n    {\n      "name": "Dr. N. Swetha Reddy",\n      "degree": "MBBS, FHRS",\n      "specialization": "Skin, Cosmetology & Hair Transplant Surgeon",\n      "registration_number": "75548"\n    }\n  ],\n  "patient": {\n    "name": "k. Leela Supreja",\n    "age": "28",\n    "sex": "F"\n  },\n  "date": "10/03/25",\n  "medicines": [\n    {\n      "name": "Manika/clanit",\n      "dosage": "6",\n      "frequency": "3 times"\n    },\n    {\n      "name": "Khuges"\n    },\n    {\n      "name": "Achin N3",\n      "form": "cap",\n      "dosage": "604",\n      

In [87]:
resp = resp.strip("```")
resp = resp.replace("json","")
resp = json.loads(resp)

In [88]:
resp["medicines"]

[{'name': 'Manika/clanit', 'dosage': '6', 'frequency': '3 times'},
 {'name': 'Khuges'},
 {'name': 'Achin N3', 'form': 'cap', 'dosage': '604', 'frequency': '1/2m'},
 {'name': 'Kerlyto', 'form': 'cap', 'frequency': '1-0-1', 'quantity': '30'},
 {'name': 'Mintop Pro', 'frequency': '2m/k'}]

In [2]:
import pandas as pd
import re
def markdown_to_dataframe(markdown_text):
    '''
    Input : pdf path => str
    description : creating 2 page chunks
    Output : list of chunks => list
    '''
    table_pattern = r"((?:\|.+\|(?:\n|\r))+\|.*\|)"
    tables = re.findall(table_pattern, markdown_text)
    combined_df = pd.DataFrame()
    for table in tables:
        rows = table.strip().split("\n")
        headers = rows[0].strip("|").split("|")
        headers = [h.strip() for h in headers]
        data = []
        len_records = []
        for row in rows[2:]:
            values = row.strip("|").split("|")
            values = [v.strip() for v in values]
            data.append(values)
            len_records.append(len(values))
        df = pd.DataFrame(data, columns=headers)
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    return combined_df

In [ ]:
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv
load_dotenv()
llm = AzureChatOpenAI(
    model = 'gpt-4o',
    api_version= '2025-01-01-preview',
    temperature=0
)

In [1]:
prompt = f"""
You are expert in extracting data from medical records. Your task is to extract the bio marker values from the medical records in a table format.
Medical Records : {document}
*The output should be only markdown table.*
*Each column name should be a bio-marker name, the column value should be bio-marker value.*
*The first column should be report date and next column should be doctor name*
"""

NameError: name 'document' is not defined

In [ ]:
ans = llm.invoke(prompt).content
ans = ans.strip("```markdown").replace("```","")

In [ ]:
print(ans)

In [ ]:
df_openai = markdown_to_dataframe(ans)

In [ ]:
df_openai.shape

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()
llm_gemini= ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
)

In [ ]:
resp = llm_gemini.invoke(prompt).content

In [ ]:
print(resp)

In [ ]:
df_gemini = markdown_to_dataframe(resp)

In [ ]:
df_gemini.shape

In [ ]:
# Global cache for column mappings
COLUMN_MAPPING_CACHE = {}

def normalize_column_names_llm(columns, llm):
    
    uncached_columns = [col for col in columns if col not in COLUMN_MAPPING_CACHE]
    
    if not uncached_columns:
        return COLUMN_MAPPING_CACHE

    prompt = f"""
You are a medical data standardization expert. Convert the following lab report column names to standard biomarker abbreviations using international nomenclature guidelines.

Rules:
1. Use only standard abbreviations (e.g., ALT, AST, HbA1c)
2. Remove units, reference ranges, and method information
3. Convert similar names to single standard (e.g., "HbA1c", "A1C" → "HbA1c")
4. For non-biomarker columns (like patient IDs), return "ignore"
5. Maintain case sensitivity (all caps for biomarkers)

Column names to normalize: {uncached_columns}

Return ONLY a JSON dictionary where:
- Key = original column name
- Value = normalized standard name or "ignore"
"""
    try:
        response = llm.invoke(prompt).content
        cleaned = response.replace("```json", "").replace("```", "").strip()
        new_mappings = json.loads(cleaned)
        
        # Cache new mappings and filter invalid
        for orig, normalized in new_mappings.items():
            if normalized.lower() != "ignore":
                COLUMN_MAPPING_CACHE[orig] = normalized
                
        return COLUMN_MAPPING_CACHE
    
    except Exception as e:
        print(f"Normalization failed: {e}")
        # Fallback to original names
        return {col: col for col in columns}

In [ ]:
from langchain.vectorstores import Weaviate
from langchain.embeddings import HuggingFaceEmbeddings
import weaviate
from langchain_weaviate.vectorstores import WeaviateVectorStore

In [ ]:
import os
from PyPDF2 import PdfReader
reader = PdfReader(file_path)
pages = reader.pages
document = []
for i in range(len(pages)):
    document.append(pages[i].extract_text())
    

### Quadrant - vector store